# AMLNet Data Quality Assessment

This notebook brings together the transaction structure checks and account balance
audit for `amlnet.transactions`, the transaction table in the BigQuery data
warehouse. Each row describes one recorded transaction: who sent and received
the funds (`origin_id`, `destination_id`), when it occurred, its `amount`, the
sender's reported balances, and the supplied fraud and money laundering labels.

The saved results cover **1,090,000 transactions**, from **13 October 2025 to
27 April 2026 (UTC)**, involving **11,000 distinct account IDs** across
`origin_id` and `destination_id`. Here, “account” means a transaction participant
identified by either column; it does not establish bank membership or a verified
customer or merchant classification.



## Purpose: assess readiness for an interactive Power BI dashboard

The goal is to build an interactive Power BI report on this data warehouse, allowing users to filter transaction activity and explore connections involving transactions labelled as fraud or money laundering. Before selecting the report's fields and measures, we need to establish which information is reliable and whether the transaction records can support an account-level balance history similar to a bank statement.

This notebook records two assessments that are critical for dashboard development:

1. **Does the merchant information consistently describe the recipient identified by `destination_id`?**  
   We compare `destination_id` with `merchant_id` to determine whether they identify the same recipient or follow a consistent mapping. We also examine whether `merchant_risk_level` is consistent for each `merchant_id` and `destination_id`. These checks establish whether merchant information can reliably be used to describe transaction recipients.

2. **Can the recorded transactions reconstruct a consistent account balance history?**  
   This requires three conditions:

   1. **The account ID must appear in `origin_id` at least once with a reported balance.**  
      The table records balances only in `origin_balance_before` and `origin_balance_after`. Therefore, an account that appears only in `destination_id` has no observed balance from which its balance history can be reconstructed.

   2. **Balances must be consistent within each transaction.**  
      Check whether `origin_balance_before - amount` agrees with `origin_balance_after`, allowing a tolerance of 0.01.

   3. **Balances must be consistent across time.**  
      Check whether an earlier reported balance, adjusted for subsequent incoming and outgoing transaction amounts, explains later reported balances within a tolerance of 0.01.

## Findings at a glance

| What was checked | Finding from the saved results | Result of the check |
|---|---|---|
| Whether `merchant_id` consistently identifies the recipient recorded in `destination_id` | Only 257 of the 293,790 transactions with a `merchant_id` have matching `merchant_id` and `destination_id` values. Each merchant ID is associated with multiple destination IDs, and vice versa. | ❌ **Fail** — `merchant_id` and `destination_id` are not interchangeable, and no consistent one-to-one mapping between them is observed. |
| Whether `merchant_risk_level` is consistent for each `merchant_id` | 943 of 1,000 merchant IDs have a single observed risk level, while 57 have multiple risk levels. The available data do not establish whether these differences represent valid changes over time or inconsistent metadata. | ⚠️ **Mixed** — risk level is consistent for most merchant IDs, but the variation observed for 57 merchants remains unexplained. |
| Whether `origin_balance_before - amount` agrees with `origin_balance_after` within each transaction | Every transaction agrees within 0.000000001, well inside the 0.01 tolerance. | ✅ **Pass** — the reported before-and-after balances are arithmetically consistent within individual transactions. |
| Whether reported account balances remain consistent across time after accounting for recorded incoming and outgoing transactions | 80.99% of comparisons between successive outgoing transactions fail the 0.01 tolerance. When reconstruction starts from each account's first reported balance, 99.52% of later comparisons fail. All 9,967 testable accounts have at least one failure. An additional 33 accounts contain tied timestamps and are excluded from these temporal comparisons. | ❌ **Fail** — reported balances cannot be reconciled consistently with the recorded transaction flows across time. |

**Conclusion:**

1. **Exclude the merchant-related fields from the dashboard dataset.**  
   `merchant_id` does not consistently identify the recipient represented by `destination_id`, and the meaning of variation in `merchant_risk_level` cannot be established from the available data. Retain `destination_id` as the recorded transaction recipient.

2. **Cancel the planned account-level dashboard based on bank-statement-style balance histories.**  
   Although the reported balances are internally consistent within individual transactions, they do not reconcile with recorded incoming and outgoing transactions across time. The balance fields therefore cannot support a reliable reconstructed account history.

Proceed with the **Transaction Overview** and **Fraud & AML Connections** pages using transaction activity, transaction counterparties, and the supplied fraud and money-laundering labels.

## Setup and execution

This project uses **Google BigQuery** as the cloud data warehouse and a Jupyter notebook as the exploratory analysis environment.

Before running this notebook, make sure that:

- BigQuery authentication has been configured for the current environment;
- the active Google Cloud project contains the `amlnet` dataset in the **EU** location; and
- the cleaned `amlnet.transactions` table has already been created using the preceding data-preparation SQL script.

The assessment is performed in a notebook rather than a standalone SQL file because the notebook provides a more suitable environment for exploratory validation. SQL queries, returned results, explanatory Markdown, and follow-up checks can be reviewed together as the analysis develops.

Some BigQuery queries are submitted through a small helper rather than relying entirely on the notebook frontend, because a notebook cell may remain in a running state even after the corresponding BigQuery job has completed. The helper submits the query as a BigQuery job and allows its actual execution status to be verified using the job information returned by BigQuery

The notebook should be run from top to bottom so that the setup, helper functions, validation queries, and resulting audit tables are created in the expected order.

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from google.cloud import bigquery

# Uncomment and set this only if needed:
# bigquery_magics.context.project = "your-project-id"

client = bigquery.Client(
    project=bigquery_magics.context.project,
    credentials=bigquery_magics.context.credentials,
)
BQ_LOCATION = "EU"
bigquery_magics.context.progress_bar_type = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
print("BigQuery project:", client.project)

pd.set_option("display.max_seq_items", 50)


c:\Users\19148\anaconda3\envs\amlnet\Lib\site-packages\google\api_core\_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
c:\Users\19148\anaconda3\envs\amlnet\Lib\site-packages\google\auth\transport\grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum 

BigQuery project: gen-lang-client-0697353147


### Register submission and completion helpers

Run this cell once per kernel session. Each SQL job has a separate name,
so a completion check cannot accidentally inspect another section's job.
A successful assertion is confirmed by a completed job with no error;
it does not need to return a table. The helper preserves SQL as written.


In [2]:
from IPython.core.magic import register_cell_magic
from uuid import uuid4

# Keep submitted job references if this helper cell is run again.
bq_jobs = globals().get("bq_jobs", {})

@register_cell_magic
def bq_submit(line, cell):
    name = line.strip()
    if not name.isidentifier():
        raise ValueError("Use a job name, for example: %%bq_submit input_checks")

    # Save and print the ID before sending, so a timeout does not lose it.
    reference = {
        "job_id": "notebook_" + uuid4().hex,
        "project": client.project,
        "location": BQ_LOCATION,
    }
    bq_jobs[name] = reference
    print(f"Submitting {name}: {reference['job_id']}", flush=True)
    client.query(
        cell,
        job_id=reference["job_id"],
        location=reference["location"],
        retry=None,
        job_retry=None,
        timeout=20,
    )
    print("Submitted. Run the completion-check cell next.")

def check_bq_job(name):
    if name not in bq_jobs:
        raise RuntimeError(f"No job reference for {name}. Run its submission cell first.")

    job = client.get_job(**bq_jobs[name], retry=None, timeout=10)
    print(f"{name}: {job.state} | {job.job_id}", flush=True)

    if job.state != "DONE":
        raise RuntimeError(
            "Job is still pending or running. Rerun ONLY this completion-check "
            "cell shortly; do not run dependent cells or resubmit the SQL."
        )
    if job.error_result is not None:
        raise RuntimeError(job.error_result.get("message", str(job.error_result)))

    print("Completed successfully.")
    return job


## Input sanity checks

The preceding data preparation script establishes the transaction table's structure, completeness, and unique transaction IDs. These are prerequisites,
not new audit questions. Run the short checks below to confirm that the expected table and input types are still available. 

In [3]:
%%bigquery --use_rest_api

WITH required_columns AS (
    SELECT column_name
    FROM UNNEST([
        'transaction_id', 'transaction_timestamp', 'transaction_date',
        'transaction_type', 'transaction_category', 'amount',
        'origin_id', 'destination_id',
        'merchant_id', 'merchant_risk_level',
        'origin_balance_before', 'origin_balance_after',
        'state', 'city', 'postcode',
        'label_is_fraud', 'label_is_money_laundering',
        'label_laundering_typology', 'provided_fraud_probability'
    ]) AS column_name
)

SELECT
    r.column_name,
    c.data_type,
    c.is_nullable,
    c.column_name IS NOT NULL AS column_exists
FROM required_columns AS r
LEFT JOIN `amlnet.INFORMATION_SCHEMA.COLUMNS` AS c
    ON c.table_name = 'transactions'
    AND c.column_name = r.column_name
ORDER BY r.column_name;


,column_name,data_type,is_nullable,column_exists
0,amount,NUMERIC,NO,True
1,city,STRING,NO,True
2,destination_id,STRING,NO,True
3,label_is_fraud,BOOL,NO,True
4,label_is_money_laundering,BOOL,NO,True
5,label_laundering_typology,STRING,YES,True
6,merchant_id,STRING,YES,True
7,merchant_risk_level,STRING,YES,True
8,origin_balance_after,NUMERIC,NO,True
9,origin_balance_before,NUMERIC,NO,True


## 1. Merchant ID and risk-level checks

Check whether `merchant_id` identifies the recipient recorded in
`destination_id`, and whether `merchant_risk_level` can be interpreted
consistently for that recipient. Compare the IDs both as sets and on individual
transactions, then examine the risk levels.


### 1.1. Compare `merchant_id` with `destination_id`

First compare merchant IDs with recipients that never appear in `origin_id`.
The query calculates both set sizes and their overlap directly. This comparison
belongs to the merchant check; it does not repeat the general account-population
analysis. Equal sets alone do not establish that the IDs agree on the same row.


In [7]:
%%bigquery --use_rest_api

WITH merchants AS (
    SELECT DISTINCT merchant_id
    FROM `amlnet.transactions`
    WHERE merchant_id IS NOT NULL
),
destination_only AS (
    SELECT destination_id
    FROM `amlnet.transactions`
    WHERE destination_id IS NOT NULL

    EXCEPT DISTINCT

    SELECT origin_id
    FROM `amlnet.transactions`
    WHERE origin_id IS NOT NULL
)

SELECT
    (SELECT COUNT(*) FROM merchants) AS n_merchant,
    (SELECT COUNT(*) FROM destination_only) AS n_destination_only,
    (
        SELECT COUNT(*)
        FROM merchants AS m
        INNER JOIN destination_only AS d
            ON m.merchant_id = d.destination_id
    ) AS n_merchant_in_destination;


,n_merchant,n_destination_only,n_merchant_in_destination
0,1000,1000,1000


**Observation**

The earlier saved results establish 1,000 distinct merchant IDs, 1,000 destination-only IDs, and 1,000 IDs in their intersection.   
The two sets contain the same identifiers.   
Next, check whether the IDs correspond on individual transactions.


#### Compare the IDs on the same transaction

Measure agreement between `destination_id` and `merchant_id` across all transactions with a non-null merchant ID. The preceding coverage check establishes how this population relates to destination-only transactions.

In [ ]:
%%bigquery --use_rest_api

SELECT
    COUNT(*) AS n_merchant_rows,

    COUNTIF(
        destination_id = merchant_id
    ) AS n_matching_rows,

    COUNTIF(
        destination_id != merchant_id
    ) AS n_mismatching_rows,

    SAFE_DIVIDE(
        COUNTIF(destination_id = merchant_id),
        COUNT(*)
    ) AS matching_share
FROM `amlnet.transactions`
WHERE merchant_id IS NOT NULL;

,n_merchant_rows,n_matching_rows,n_mismatching_rows,matching_share
0,293790,257,293533,0.000875


**Observation**

Only 257 of 293,790 transactions with merchant metadata have matching IDs (approximately 0.0875%); 293,533 have different IDs. Despite the set-level overlap, `merchant_id` cannot be treated as an interchangeable copy of the transaction destination. Different values alone do not rule out a separate, consistent mapping, so the next check examines mapping cardinality.

#### Check mapping cardinality within the destination-only population

For transactions whose destinations never appear as origins, count how many distinct merchant IDs map to each destination and how many distinct destinations map to each merchant ID. Summarize both directions. Values greater than one identify departures from a one-to-one mapping; missing merchant IDs are covered by the preceding coverage check.

In [ ]:
%%bigquery --use_rest_api

WITH destination_only AS (
    SELECT destination_id
    FROM `amlnet.transactions`
    WHERE destination_id IS NOT NULL

    EXCEPT DISTINCT

    SELECT origin_id
    FROM `amlnet.transactions`
    WHERE origin_id IS NOT NULL
),
merchant_pairs AS (
    SELECT DISTINCT
        t.destination_id,
        t.merchant_id
    FROM `amlnet.transactions` AS t
    INNER JOIN destination_only AS d
        ON t.destination_id = d.destination_id
    WHERE t.merchant_id IS NOT NULL
),
mapping_counts AS (
    SELECT
        'destination_id -> merchant_id' AS mapping_direction,
        destination_id AS entity_id,
        COUNT(*) AS n_linked_ids
    FROM merchant_pairs
    GROUP BY destination_id

    UNION ALL

    SELECT
        'merchant_id -> destination_id' AS mapping_direction,
        merchant_id AS entity_id,
        COUNT(*) AS n_linked_ids
    FROM merchant_pairs
    GROUP BY merchant_id
)

SELECT
    mapping_direction,
    COUNT(*) AS n_entities,
    COUNTIF(n_linked_ids = 1) AS n_with_one_linked_id,
    COUNTIF(n_linked_ids > 1) AS n_with_multiple_linked_ids,
    MAX(n_linked_ids) AS max_linked_ids
FROM mapping_counts
GROUP BY mapping_direction
ORDER BY mapping_direction;

,mapping_direction,n_entities,n_with_one_linked_id,n_with_multiple_linked_ids,max_linked_ids
0,destination_id -> merchant_id,1000,0,1000,305
1,merchant_id -> destination_id,1000,0,1000,302


**Observation**

All 1,000 IDs in each mapping direction have multiple linked IDs. A single
destination is paired with as many as 305 merchant IDs; a merchant ID is
paired with as many as 302 destinations. This is a many-to-many mapping,
rather than a stable one-to-one substitution between identifiers.


#### Inspect example ID mismatches

Inspect transactions with different destination and merchant IDs alongside their transaction type and category. These examples illustrate the aggregate finding; they are not a representative sample or evidence of fraud.

In [ ]:
%%bigquery --use_rest_api

SELECT
    destination_id,
    merchant_id,
    transaction_type,
    transaction_category
FROM `amlnet.transactions`
WHERE
    merchant_id IS NOT NULL
    AND destination_id != merchant_id
LIMIT 20;

,destination_id,merchant_id,transaction_type,transaction_category
0,M42,M982,OSKO,Education
1,M431,M505,DEBIT,Education
2,M464,M468,DEBIT,Education
3,M841,M135,TRANSFER,Education
4,M528,M669,TRANSFER,Education
5,M830,M924,TRANSFER,Education
6,M431,M922,TRANSFER,Education
7,M475,M585,TRANSFER,Education
8,M160,M93,BPAY,Education
9,M700,M363,TRANSFER,Education


**Observation**

The saved examples include destination `M431` paired with merchant IDs `M505` and `M922`, and destination `M475` paired with `M585` and `M631`. A destination therefore does not consistently identify one metadata merchant in these examples. The mapping-cardinality query above quantifies this issue across the destination-only population.

### 1.2. Check `merchant_risk_level` consistency

First group by `merchant_id`, then summarize how many IDs have zero, one, or multiple distinct non-null risk levels. A count of zero means all risk levels for that ID are null. Multiple levels mean the field cannot be used as a single stable merchant classification without a justified rule for interpreting the variation.

In [ ]:
%%bigquery --use_rest_api

SELECT
    n_risk_levels,
    COUNT(*) AS n_merchants
FROM (
    SELECT
        merchant_id,
        COUNT(DISTINCT merchant_risk_level) AS n_risk_levels
    FROM `amlnet.transactions`
    WHERE merchant_id IS NOT NULL
    GROUP BY merchant_id
)
GROUP BY n_risk_levels
ORDER BY n_risk_levels;

,n_risk_levels,n_merchants
0,1,943
1,2,56
2,3,1


**Observation**

Of the 1,000 merchant IDs, 943 have one observed non-null risk level, 56 have two, and 1 has three. Thus, 57 IDs (5.7%) have multiple risk levels. This demonstrates variation within merchant IDs, but does not by itself distinguish legitimate changes over time from inconsistent metadata.

#### Check risk levels using the destination as the entity key

Repeat the risk-level check using `destination_id` on transactions with a merchant ID. This tests whether substituting the actual transaction destination would resolve the lack of a stable risk classification.

In [ ]:
%%bigquery --use_rest_api

SELECT
    destination_id,
    COUNT(DISTINCT merchant_risk_level) AS n_risk_levels
FROM `amlnet.transactions`
WHERE merchant_id IS NOT NULL
GROUP BY destination_id
ORDER BY n_risk_levels DESC;

,destination_id,n_risk_levels
0,M335,2
1,M355,2
2,M439,2
3,M423,2
4,M168,2
...,...,...
995,M111,1
996,M558,1
997,M382,1
998,M653,1


**Observation**

The saved result also contains destinations with multiple risk levels. Replacing `merchant_id` with `destination_id` therefore does not produce a single stable risk classification for every entity. These checks do not establish a validated historical risk model that would make the variation suitable for dashboard use.

### 1.3. Decision on merchant fields

Exclude merchant metadata from dashboard enrichment. The ID sets coincide, but
same-row IDs agree for only 257 of 293,790 transactions (0.0875%). Every one of
the 1,000 destination IDs and 1,000 merchant IDs has multiple partners in the
mapping. The observed correspondence cannot supply a one-to-one merchant lookup.

Risk levels vary for 57 merchant IDs (5.7%). Variation alone is not proof of bad
data: risk can change over time. These checks do not establish a validated
historical risk model or explain the many-to-many transaction assignments.

Do not select an arbitrary first, most frequent, or highest-risk merchant value
to manufacture a profile. Retain `destination_id` as the recorded recipient.
The exclusion covers merchant metadata as a group; fields such as
`merchant_avg_transaction` were not independently validated in these notebooks.
These are suitability findings for this project, not evidence of fraud or AML.


## 2. Account balance history checks

Test whether `origin_balance_before` and `origin_balance_after` agree with `amount` within each transaction, then whether the balances agree across time. For the second check, use accounts that appear in `origin_id` at least once, because only those accounts have a reported reference balance. When reconstructing each account's balance over time, account for both outgoing transactions, where the account appears as `origin_id`, and incoming transactions, where it appears as `destination_id`.

### 2.1. Prepare the accounts and their transaction histories

Create one row per account appearing in `origin_id`, named `entity_id` in the
derived tables. This account list is required by the history-building queries.
It is restricted to the balance audit and must not define the dashboard population.


In [ ]:
%%bq_submit create_account_population

CREATE OR REPLACE TABLE `amlnet.account_population`
CLUSTER BY entity_id
AS
SELECT DISTINCT origin_id AS entity_id
FROM `amlnet.transactions`;

Submitting create_account_population: notebook_6a29d8b233364a62ba411b82695c7687
Submitted. Run the completion-check cell next.


#### Confirm completion: `create_account_population`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [ ]:
create_account_population_job = check_bq_job("create_account_population")


create_account_population: DONE | notebook_6a29d8b233364a62ba411b82695c7687
Completed successfully.


#### Build incoming and outgoing transaction histories

Transform the transaction table into an **account-history-style dataset centered on `entity_id`** (accounts that appear in `origin_id` at least once). For each account, collect transactions in which it appears as either origin or destination, recording its role and transaction timestamp so that its history can be examined chronologically.

In the original dataset, `origin_balance_before` and `origin_balance_after` are recorded only for origin accounts. Restricting the `entity_id` population therefore does not affect either balance consistency check:

1. **Within-transaction consistency:** Every transaction retains its origin account and the corresponding balance information, so no transactions are excluded from this check.
2. **Across-time consistency:** Accounts appearing only in `destination_id` have no reported reference balance and therefore cannot satisfy the prerequisite for reconstructing their balance histories.



In [23]:
%%bq_submit create_entity_transactions

CREATE OR REPLACE TABLE `amlnet.entity_transactions`
PARTITION BY transaction_date
CLUSTER BY entity_id, transaction_role
AS
SELECT
    t.transaction_id,
    t.transaction_timestamp,
    t.transaction_date,
    t.origin_id AS entity_id,
    'origin' AS transaction_role,
    t.destination_id AS counterparty_id,
    a.entity_id IS NOT NULL AS counterparty_in_entity_population,
    t.origin_id = t.destination_id AS is_self_transfer,
    t.transaction_type,
    t.transaction_category,
    t.amount,
    -t.amount AS signed_amount,
    t.origin_balance_before AS balance_before,
    t.origin_balance_after AS balance_after,
    t.origin_balance_before IS NOT NULL
        AND t.origin_balance_after IS NOT NULL AS balance_observed,
    t.state AS transaction_state,
    t.city AS transaction_city,
    t.postcode AS transaction_postcode,
    t.label_is_fraud,
    t.label_is_money_laundering,
    t.label_laundering_typology,
    t.provided_fraud_probability
FROM `amlnet.transactions` AS t
LEFT JOIN `amlnet.account_population` AS a
    ON t.destination_id = a.entity_id

UNION ALL

SELECT
    t.transaction_id,
    t.transaction_timestamp,
    t.transaction_date,
    t.destination_id AS entity_id,
    'destination' AS transaction_role,
    t.origin_id AS counterparty_id,
    TRUE AS counterparty_in_entity_population,
    t.origin_id = t.destination_id AS is_self_transfer,
    t.transaction_type,
    t.transaction_category,
    t.amount,
    t.amount AS signed_amount,
    CAST(NULL AS NUMERIC) AS balance_before,
    CAST(NULL AS NUMERIC) AS balance_after,
    FALSE AS balance_observed,
    t.state AS transaction_state,
    t.city AS transaction_city,
    t.postcode AS transaction_postcode,
    t.label_is_fraud,
    t.label_is_money_laundering,
    t.label_laundering_typology,
    t.provided_fraud_probability
FROM `amlnet.transactions` AS t
INNER JOIN `amlnet.account_population` AS a
    ON t.destination_id = a.entity_id;


Submitting create_entity_transactions: notebook_dbb2ab44e7c04755a9238d6f510d1e1c
Submitted. Run the completion-check cell next.


#### Confirm completion: `create_entity_transactions`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [14]:
create_entity_transactions_job = check_bq_job("create_entity_transactions")


create_entity_transactions: DONE | notebook_ebae8a4059a5480eba1f6e6189b35db9
Completed successfully.


#### Preview transformed transaction records

After table creation succeeds, inspect the first 20 role rows.


In [24]:
%%bigquery --use_rest_api

SELECT *
FROM `amlnet.entity_transactions`
ORDER BY transaction_timestamp, transaction_id, transaction_role
LIMIT 20;


,transaction_id,transaction_timestamp,transaction_date,entity_id,transaction_role,counterparty_id,counterparty_in_entity_population,is_self_transfer,transaction_type,transaction_category,amount,signed_amount,balance_before,balance_after,balance_observed,transaction_state,transaction_city,transaction_postcode,label_is_fraud,label_is_money_laundering,label_laundering_typology,provided_fraud_probability
0,1804902,2025-10-13 17:08:30.636449+00:00,2025-10-13,C8198,destination,C4583,True,False,TRANSFER,Recreation,203.977370750,203.977370750,None,None,False,SA,Adelaide,5949,False,False,normal,None
1,1804902,2025-10-13 17:08:30.636449+00:00,2025-10-13,C4583,origin,C8198,True,False,TRANSFER,Recreation,203.977370750,-203.977370750,298829.722273968,298625.744903218,True,SA,Adelaide,5949,False,False,normal,None
2,1681023,2025-10-13 17:08:34.636449+00:00,2025-10-13,C873,destination,C8577,True,False,DEBIT,Transport,143.220633449,143.220633449,None,None,False,VIC,Melbourne,3784,False,False,normal,None
3,1681023,2025-10-13 17:08:34.636449+00:00,2025-10-13,C8577,origin,C873,True,False,DEBIT,Transport,143.220633449,-143.220633449,402841.306482348,402698.085848899,True,VIC,Melbourne,3784,False,False,normal,None
4,1983078,2025-10-13 17:08:37.636449+00:00,2025-10-13,C8962,origin,M382,False,False,DEBIT,Housing,4323.773401516,-4323.773401516,269354.771448464,265030.998046948,True,SA,Adelaide,5059,False,False,normal,None
5,1179643,2025-10-13 17:08:54.636449+00:00,2025-10-13,C5339,destination,C3579,True,False,BPAY,Transport,99.679952814,99.679952814,None,None,False,QLD,Brisbane,4564,False,False,normal,None
6,1179643,2025-10-13 17:08:54.636449+00:00,2025-10-13,C3579,origin,C5339,True,False,BPAY,Transport,99.679952814,-99.679952814,459083.273778897,458983.593826083,True,QLD,Brisbane,4564,False,False,normal,None
7,1817565,2025-10-13 17:09:00.636449+00:00,2025-10-13,C7149,destination,C2756,True,False,BPAY,Recreation,124.683942704,124.683942704,None,None,False,VIC,Melbourne,3480,False,False,normal,None
8,1817565,2025-10-13 17:09:00.636449+00:00,2025-10-13,C2756,origin,C7149,True,False,BPAY,Recreation,124.683942704,-124.683942704,205161.239523179,205036.555580475,True,VIC,Melbourne,3480,False,False,normal,None
9,1410830,2025-10-13 17:09:05.636449+00:00,2025-10-13,C4320,destination,C81,True,False,DEBIT,Food,202.391155030,202.391155030,None,None,False,NSW,Sydney,2911,False,False,normal,None


#### Summarize the history

Summarize the account histories by transaction role.

In [19]:
%%bigquery --use_rest_api

SELECT
    transaction_role,
    COUNT(*) AS history_rows,
    COUNT(DISTINCT entity_id) AS accounts,
    SUM(amount) AS amount,
    SUM(signed_amount) AS signed_amount,
    COUNTIF(balance_observed) AS balance_observations
FROM `amlnet.entity_transactions`
GROUP BY transaction_role;


,transaction_role,history_rows,accounts,amount,signed_amount,balance_observations
0,origin,1090000,10000,707459101.937771781,-707459101.937771781,1090000
1,destination,796210,9995,139567384.532919084,139567384.532919084,0


**Observation**

The audit history contains 1,090,000 origin rows and 796,210 destination rows: 1,886,210 role rows in total. Balance observations are available on all origin rows and on no destination rows, as intended.


### 2.2. Within-transaction balance arithmetic

Check whether `origin_balance_before` - `amount` agrees with `origin_balance_after` for each transaction. Calculate the difference between the recorded and expected balances, and examine both exact matches and matches within a tolerance of 0.01.

In [ ]:
%%bq_submit create_transaction_balance_audit

CREATE OR REPLACE TABLE `amlnet.transaction_balance_audit`
PARTITION BY transaction_date
CLUSTER BY entity_id, transaction_status
AS
WITH differences AS (
    SELECT
        transaction_id,
        transaction_timestamp,
        transaction_date,
        entity_id,
        transaction_type,
        amount,
        balance_before,
        balance_after,
        is_self_transfer,
        label_is_fraud,
        label_is_money_laundering,
        IF(is_self_transfer, NULL, balance_before - amount)
            AS expected_balance_after,
        IF(is_self_transfer, NULL, balance_after - (balance_before - amount))
            AS balance_difference
    FROM `amlnet.entity_transactions`
    WHERE transaction_role = 'origin'
)

SELECT
    *,
    CASE
        WHEN is_self_transfer THEN 'self_transfer_unresolved'
        WHEN balance_before IS NULL OR balance_after IS NULL
            THEN 'missing_balance'
        WHEN balance_difference = 0 THEN 'exact_match'
        WHEN ABS(balance_difference) <= NUMERIC '0.01'
            THEN 'within_one_cent'
        ELSE 'mismatch'
    END AS transaction_status
FROM differences;


Submitting create_transaction_balance_audit: notebook_b5115eb9fde84ce09212cefa7ed6b5c7
Submitted. Run the completion-check cell next.


#### Confirm completion: `create_transaction_balance_audit`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [ ]:
create_transaction_balance_audit_job = check_bq_job("create_transaction_balance_audit")


create_transaction_balance_audit: DONE | notebook_b5115eb9fde84ce09212cefa7ed6b5c7
Completed successfully.


#### Summarize transaction balance statuses

After the creation job succeeds, display the audit status counts.


In [ ]:
%%bigquery --use_rest_api

SELECT
    transaction_status,
    COUNT(*) AS transactions,
    COUNT(DISTINCT entity_id) AS accounts,
    MAX(ABS(balance_difference)) AS max_absolute_difference
FROM `amlnet.transaction_balance_audit`
GROUP BY transaction_status
ORDER BY transactions DESC;

,transaction_status,transactions,accounts,max_absolute_difference
0,exact_match,817594,10000,0E-9
1,within_one_cent,272406,10000,1E-9


**Observation**

All 1,090,000 debit transitions reconcile within 0.000000001. There are 817,594
exact matches (75.01%) and 272,406 residuals of only 0.000000001 (24.99%). These
tiny differences are compatible with rounding; there are no substantive
within-transaction discrepancies.


### 2.3. Temporal audit coverage

At least two origin events are required for a later recorded balance comparison.
Earlier incoming events stay in the history without requiring an inferred opening
balance. Accounts with any same-timestamp distinct transactions or self-transfers
are excluded from temporal comparisons in this conservative audit.

A transaction ID provides reproducible display order, not evidence of the order
in which balances were posted. Exclusions remain visible in the summary; they
are not counted as passing observations.


In [ ]:
%%bigquery --use_rest_api

WITH event_times AS (
    SELECT
        entity_id,
        transaction_timestamp,
        COUNT(DISTINCT transaction_id) AS transactions_at_timestamp
    FROM `amlnet.entity_transactions`
    GROUP BY entity_id, transaction_timestamp
),
time_flags AS (
    SELECT entity_id,
        LOGICAL_OR(transactions_at_timestamp > 1) AS has_timestamp_ties
    FROM event_times
    GROUP BY entity_id
),
account_counts AS (
    SELECT entity_id,
        COUNTIF(transaction_role = 'origin') AS origin_transactions,
        COUNTIF(transaction_role = 'destination') AS destination_transactions,
        LOGICAL_OR(is_self_transfer) AS has_self_transfer
    FROM `amlnet.entity_transactions`
    GROUP BY entity_id
)

SELECT
    c.origin_transactions >= 2 AS has_multiple_outflows,
    f.has_timestamp_ties,
    c.has_self_transfer,
    COUNT(*) AS accounts,
    SUM(c.origin_transactions) AS origin_transactions,
    SUM(c.destination_transactions) AS destination_transactions
FROM account_counts AS c
INNER JOIN time_flags AS f USING (entity_id)
GROUP BY has_multiple_outflows, has_timestamp_ties, has_self_transfer
ORDER BY has_multiple_outflows DESC, has_timestamp_ties, has_self_transfer;


,has_multiple_outflows,has_timestamp_ties,has_self_transfer,accounts,origin_transactions,destination_transactions
0,True,False,False,9967,1085012,792005
1,True,True,False,33,4988,4205


**Observation**

All 10,000 audit entities have multiple outflows. Timestamp ties exclude 33
entities (0.33%) with 4,988 origin transactions. The other 9,967 entities are
temporally testable. No self-transfers are observed.


### 2.4. Temporal balance reconciliation

All incoming and outgoing events are used in the calculation; only origin events have recorded balances to compare.

The first origin event supplies a **recorded post-transaction anchor**. At each later origin event:

`theoretical_balance_before = anchor_balance_after + subsequent inflows − subsequent outflows`

Here “subsequent” excludes both the anchor transaction and the current transaction. Subtracting the current amount gives the theoretical balance after the current transaction. This trajectory never resets to a later recorded balance, so cumulative discrepancies remain visible.

A second comparison uses the **previous origin's recorded balance after** plus intervening flows. This isolates gaps between successive balance observations instead of carrying every earlier discrepancy forward. It can still be tested if an earlier anchor balance is missing.


In [ ]:
%%bq_submit create_entity_balance_audit

CREATE OR REPLACE TABLE `amlnet.entity_balance_audit`
PARTITION BY transaction_date
CLUSTER BY entity_id, rolling_status
AS
WITH event_flows AS (
    -- Collapse role rows to one account event, including self-transfers.
    SELECT
        entity_id,
        transaction_id,
        transaction_timestamp,
        SUM(signed_amount) AS net_amount
    FROM `amlnet.entity_transactions`
    GROUP BY entity_id, transaction_id, transaction_timestamp
),
event_windows AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY entity_id, transaction_timestamp
        ) AS events_at_timestamp,
        COALESCE(SUM(net_amount) OVER (
            PARTITION BY entity_id
            ORDER BY transaction_timestamp, transaction_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), NUMERIC '0') AS cumulative_before,
        SUM(net_amount) OVER (
            PARTITION BY entity_id
            ORDER BY transaction_timestamp, transaction_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_after
    FROM event_flows
),
account_flags AS (
    SELECT entity_id,
        LOGICAL_OR(events_at_timestamp > 1) AS has_timestamp_ties
    FROM event_windows
    GROUP BY entity_id
),
self_flags AS (
    SELECT entity_id,
        LOGICAL_OR(is_self_transfer) AS has_self_transfer
    FROM `amlnet.entity_transactions`
    GROUP BY entity_id
),
origin_events AS (
    SELECT
        o.*,
        e.cumulative_before,
        e.cumulative_after,
        f.has_timestamp_ties,
        s.has_self_transfer
    FROM `amlnet.transaction_balance_audit` AS o
    INNER JOIN event_windows AS e
        USING (entity_id, transaction_id, transaction_timestamp)
    INNER JOIN account_flags AS f USING (entity_id)
    INNER JOIN self_flags AS s USING (entity_id)
),
anchors AS (
    SELECT
        *,
        ROW_NUMBER() OVER origin_order AS origin_number,
        COUNT(*) OVER (PARTITION BY entity_id) AS origin_transaction_count,
        FIRST_VALUE(transaction_id) OVER origin_order AS anchor_transaction_id,
        FIRST_VALUE(balance_after) OVER origin_order AS anchor_balance_after,
        FIRST_VALUE(cumulative_after) OVER origin_order AS anchor_cumulative_after,
        LAG(transaction_id) OVER origin_order AS previous_origin_transaction_id,
        LAG(balance_after) OVER origin_order AS previous_recorded_balance_after,
        LAG(cumulative_after) OVER origin_order AS previous_cumulative_after
    FROM origin_events
    WINDOW origin_order AS (
        PARTITION BY entity_id
        ORDER BY transaction_timestamp, transaction_id
    )
),
expectations AS (
    SELECT
        *,
        CASE
            WHEN NOT has_timestamp_ties AND NOT has_self_transfer
                AND origin_number > 1
            THEN anchor_balance_after + cumulative_before - anchor_cumulative_after
        END AS theoretical_balance_before,
        CASE
            WHEN NOT has_timestamp_ties AND NOT has_self_transfer
                AND origin_number > 1
            THEN previous_recorded_balance_after
                + cumulative_before - previous_cumulative_after
        END AS expected_before_from_previous
    FROM anchors
),
differences AS (
    SELECT
        *,
        theoretical_balance_before - amount AS theoretical_balance_after,
        balance_before - theoretical_balance_before AS rolling_difference_before,
        balance_after - (theoretical_balance_before - amount)
            AS rolling_difference_after,
        balance_before - expected_before_from_previous AS interval_difference
    FROM expectations
)

SELECT
    entity_id,
    transaction_id,
    transaction_timestamp,
    transaction_date,
    transaction_type,
    amount,
    balance_before,
    balance_after,
    transaction_status,
    origin_number,
    origin_transaction_count,
    has_timestamp_ties,
    has_self_transfer,
    anchor_transaction_id,
    anchor_balance_after,
    previous_origin_transaction_id,
    previous_recorded_balance_after,
    theoretical_balance_before,
    theoretical_balance_after,
    rolling_difference_before,
    rolling_difference_after,
    expected_before_from_previous,
    interval_difference,
    CASE
        WHEN has_self_transfer THEN 'self_transfer_unresolved'
        WHEN has_timestamp_ties THEN 'ambiguous_order'
        WHEN origin_number = 1 THEN 'anchor'
        WHEN anchor_balance_after IS NULL THEN 'missing_anchor_balance'
        WHEN balance_before IS NULL OR balance_after IS NULL THEN 'missing_balance'
        WHEN rolling_difference_before = 0 AND rolling_difference_after = 0
            THEN 'exact_match'
        WHEN ABS(rolling_difference_before) <= NUMERIC '0.01'
            AND ABS(rolling_difference_after) <= NUMERIC '0.01'
            THEN 'within_one_cent'
        ELSE 'mismatch'
    END AS rolling_status,
    CASE
        WHEN has_self_transfer THEN 'self_transfer_unresolved'
        WHEN has_timestamp_ties THEN 'ambiguous_order'
        WHEN origin_number = 1 THEN 'anchor'
        WHEN previous_recorded_balance_after IS NULL THEN 'missing_previous_balance'
        WHEN balance_before IS NULL THEN 'missing_balance'
        WHEN interval_difference = 0 THEN 'exact_match'
        WHEN ABS(interval_difference) <= NUMERIC '0.01' THEN 'within_one_cent'
        ELSE 'mismatch'
    END AS interval_status
FROM differences;


Submitting create_entity_balance_audit: notebook_ba4d739fcb0d4c98af5996ddab4a2a9b
Submitted. Run the completion-check cell next.


#### Confirm completion: `create_entity_balance_audit`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [ ]:
create_entity_balance_audit_job = check_bq_job("create_entity_balance_audit")


create_entity_balance_audit: DONE | notebook_ba4d739fcb0d4c98af5996ddab4a2a9b
Completed successfully.


#### Validate temporal row coverage

Submit the assertion separately from the status summary below.


In [ ]:
%%bq_submit temporal_row_check

ASSERT (
    SELECT COUNT(*) FROM `amlnet.entity_balance_audit`
) = (
    SELECT COUNT(*) FROM `amlnet.transactions`
) AS 'Temporal audit must retain every origin transaction.';


Submitting temporal_row_check: notebook_c196f4fe93e7411c87a4a33f20bd67b0
Submitted. Run the completion-check cell next.


#### Confirm completion: `temporal_row_check`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [ ]:
temporal_row_check_job = check_bq_job("temporal_row_check")


temporal_row_check: DONE | notebook_c196f4fe93e7411c87a4a33f20bd67b0
Completed successfully.


#### Summarize temporal audit statuses

After the row-count assertion passes, summarize the temporal audit output.


In [ ]:
%%bigquery --use_rest_api

SELECT
    rolling_status,
    COUNT(*) AS origin_transactions,
    COUNT(DISTINCT entity_id) AS accounts
FROM `amlnet.entity_balance_audit`
GROUP BY rolling_status
ORDER BY origin_transactions DESC;


,rolling_status,origin_transactions,accounts
0,mismatch,1069905,9967
1,anchor,9967,9967
2,ambiguous_order,4988,33
3,exact_match,3073,2090
4,within_one_cent,2067,1063


#### Summarize temporal agreement

The rolling test checks both before and after against the first anchor. The
interval test checks only the next recorded balance before against the previous
recorded balance after plus intervening flows. Rates use testable comparisons
only; exclusions and anchor rows are reported separately. NULL rates mean
there were no testable comparisons.


In [ ]:
%%bigquery --use_rest_api

WITH audit_results AS (
    SELECT
        'first_anchor_roll_forward' AS audit,
        entity_id,
        rolling_status AS status,
        GREATEST(ABS(rolling_difference_before), ABS(rolling_difference_after))
            AS absolute_difference
    FROM `amlnet.entity_balance_audit`

    UNION ALL

    SELECT
        'between_consecutive_origins' AS audit,
        entity_id,
        interval_status AS status,
        ABS(interval_difference) AS absolute_difference
    FROM `amlnet.entity_balance_audit`
)

SELECT
    audit,
    COUNT(*) AS origin_rows,
    COUNTIF(status = 'anchor') AS anchor_rows,
    COUNTIF(status IN ('exact_match', 'within_one_cent', 'mismatch'))
        AS testable_comparisons,
    COUNTIF(status NOT IN ('anchor', 'exact_match', 'within_one_cent', 'mismatch'))
        AS excluded_rows,
    COUNT(DISTINCT IF(
        status IN ('exact_match', 'within_one_cent', 'mismatch'), entity_id, NULL
    )) AS accounts_with_testable_comparisons,
    COUNTIF(status = 'mismatch') AS mismatches,
    SAFE_DIVIDE(
        COUNTIF(status = 'exact_match'),
        COUNTIF(status IN ('exact_match', 'within_one_cent', 'mismatch'))
    ) AS exact_match_rate,
    SAFE_DIVIDE(
        COUNTIF(status IN ('exact_match', 'within_one_cent')),
        COUNTIF(status IN ('exact_match', 'within_one_cent', 'mismatch'))
    ) AS match_rate_within_one_cent,
    MAX(IF(
        status IN ('exact_match', 'within_one_cent', 'mismatch'),
        absolute_difference, NULL
    )) AS max_absolute_difference
FROM audit_results
GROUP BY audit;


,audit,origin_rows,anchor_rows,testable_comparisons,excluded_rows,accounts_with_testable_comparisons,mismatches,exact_match_rate,match_rate_within_one_cent,max_absolute_difference
0,first_anchor_roll_forward,1090000,9967,1075045,4988,9967,1069905,0.002858,0.004781,328539.466958272
1,between_consecutive_origins,1090000,9967,1075045,4988,9967,870699,0.190081,0.190081,325138.482306509


**Observation**

Of 1,075,045 testable later-origin comparisons, 1,069,905 (99.52%) fail the
first-anchor roll-forward test and 870,699 (80.99%) fail the comparison with the
previous recorded origin balance. The latter resets its reference at each origin
observation, showing that cumulative error propagation alone does not explain
the high failure rate. Maximum discrepancies are approximately 328,539.47 and
325,138.48, respectively, in the dataset's monetary units.


#### Summarize affected accounts

Distinguish untestable accounts from accounts with tested observations that
agree or disagree. One outflow supplies an anchor but no temporal evidence.
This summary is for input quality; it is not a suspicious-account ranking.


In [ ]:
%%bigquery --use_rest_api

WITH account_results AS (
    SELECT
        entity_id,
        MAX(origin_transaction_count) AS origin_transactions,
        LOGICAL_OR(has_timestamp_ties) AS has_timestamp_ties,
        LOGICAL_OR(has_self_transfer) AS has_self_transfer,
        COUNTIF(rolling_status IN ('exact_match', 'within_one_cent', 'mismatch'))
            AS tested_comparisons,
        COUNTIF(rolling_status = 'mismatch') AS rolling_mismatches,
        COUNTIF(interval_status = 'mismatch') AS interval_mismatches
    FROM `amlnet.entity_balance_audit`
    GROUP BY entity_id
),
classified AS (
    SELECT
        *,
        CASE
            WHEN has_self_transfer THEN 'self_transfer_unresolved'
            WHEN has_timestamp_ties THEN 'ambiguous_order'
            WHEN origin_transactions = 1 THEN 'one_outflow_only'
            WHEN tested_comparisons = 0 THEN 'no_testable_comparisons'
            WHEN rolling_mismatches > 0 THEN 'has_rolling_mismatch'
            WHEN tested_comparisons < origin_transactions - 1
                THEN 'tested_rows_agree_incomplete_coverage'
            ELSE 'all_later_rows_agree_within_one_cent'
        END AS account_audit_status
    FROM account_results
)

SELECT
    account_audit_status,
    COUNT(*) AS accounts,
    SUM(tested_comparisons) AS tested_comparisons,
    SUM(rolling_mismatches) AS rolling_mismatches,
    SUM(interval_mismatches) AS interval_mismatches
FROM classified
GROUP BY account_audit_status
ORDER BY accounts DESC;


,account_audit_status,accounts,tested_comparisons,rolling_mismatches,interval_mismatches
0,has_rolling_mismatch,9967,1075045,1069905,870699
1,ambiguous_order,33,0,0,0


**Observation**

Every one of the 9,967 tested entities has at least one rolling mismatch. The 33
excluded entities are unassessed, not validated. Temporal reconciliation failure
is widespread and persists after excluding accounts with ambiguous timestamps.


#### Inspect temporal discrepancies

Compare cumulative and local discrepancies side by side. A rolling mismatch
with a matching interval can reflect an earlier discrepancy carried forward.
A new interval mismatch locates a disagreement between successive observations.
Neither establishes whether the cause is synthetic data generation, unobserved
flows, transaction semantics, or erroneous balances.


In [21]:
%%bigquery --use_rest_api

SELECT
    entity_id,
    transaction_id,
    transaction_timestamp,
    origin_number,
    amount,
    anchor_balance_after,
    balance_before,
    theoretical_balance_before,
    rolling_difference_before,
    balance_after,
    theoretical_balance_after,
    rolling_difference_after,
    expected_before_from_previous,
    interval_difference,
    transaction_status,
    rolling_status,
    interval_status
FROM `amlnet.entity_balance_audit`
WHERE rolling_status = 'mismatch' OR interval_status = 'mismatch'
ORDER BY GREATEST(
    COALESCE(ABS(rolling_difference_before), NUMERIC '0'),
    COALESCE(ABS(rolling_difference_after), NUMERIC '0'),
    COALESCE(ABS(interval_difference), NUMERIC '0')
) DESC, entity_id, transaction_timestamp, transaction_id
LIMIT 20;


,entity_id,transaction_id,transaction_timestamp,origin_number,amount,anchor_balance_after,balance_before,theoretical_balance_before,rolling_difference_before,balance_after,theoretical_balance_after,rolling_difference_after,expected_before_from_previous,interval_difference,transaction_status,rolling_status,interval_status
0,C2071,1337745,2026-01-23 03:24:35.636449+00:00,54,50.478806023,466275.842912175,117452.459009576,445991.925967847,-328539.466958271,117401.980203552,445941.447161824,-328539.466958272,440621.125363886,-323168.666354310,within_one_cent,mismatch,mismatch
1,C2071,1492556,2026-01-23 04:00:33.636449+00:00,55,46.438014007,466275.842912175,117498.897023583,445941.447161824,-328442.550138241,117452.459009576,445895.009147817,-328442.550138241,117401.980203552,96.916820031,exact_match,mismatch,mismatch
2,C2071,1389460,2026-01-23 19:16:03.636449+00:00,56,1241.947862373,466275.842912175,120963.790844571,446130.463362161,-325166.672517590,119721.842982198,444888.515499788,-325166.672517590,117687.913223920,3275.877620651,exact_match,mismatch,mismatch
3,C2071,1359714,2026-01-23 21:04:33.636449+00:00,57,161.818006535,466275.842912175,119721.842982198,444888.515499788,-325166.672517590,119560.024975663,444726.697493253,-325166.672517590,119721.842982198,0E-9,exact_match,mismatch,exact_match
4,C2071,1849162,2026-01-26 05:07:45.636449+00:00,58,2061.127952080,466275.842912175,119560.024975663,444726.697493253,-325166.672517590,117498.897023583,442665.569541173,-325166.672517590,119560.024975663,0E-9,exact_match,mismatch,exact_match
5,C5346,1248103,2026-01-20 23:30:23.636449+00:00,142,94.419328721,438480.980405408,34254.616083890,359335.894309270,-325081.278225380,34160.196755169,359241.474980549,-325081.278225380,359393.098390399,-325138.482306509,exact_match,mismatch,mismatch
6,C5346,1922169,2026-01-21 01:33:33.636449+00:00,143,64.890731030,438480.980405408,34319.506814920,359241.474980549,-324921.968165629,34254.616083890,359176.584249519,-324921.968165629,34160.196755169,159.310059751,exact_match,mismatch,mismatch
7,C5346,1601977,2026-01-21 10:46:03.636449+00:00,144,39.643075853,438480.980405408,34359.149890774,359251.910558786,-324892.760668012,34319.506814920,359212.267482933,-324892.760668013,34329.942393157,29.207497617,within_one_cent,mismatch,mismatch
8,C2071,1468546,2026-01-26 10:54:54.636449+00:00,59,72488.247118552,466275.842912175,440533.437865259,442665.569541173,-2132.131675914,368045.190746706,370177.322422621,-2132.131675915,117498.897023583,323034.540841676,within_one_cent,mismatch,mismatch
9,C5346,1768363,2026-01-23 08:10:34.636449+00:00,145,3644.014747927,438480.980405408,349838.534683754,359212.267482933,-9373.732799179,346194.519935827,355568.252735006,-9373.732799179,34319.506814920,315519.027868834,exact_match,mismatch,mismatch


### 2.5. Inspect a discrepant account history

By default, select the account with the most temporal mismatches, breaking ties
by ID. Set `selected_entity_id` to an actual account ID to inspect another one.
Both incoming and outgoing events are shown, including events before the anchor.
Destination balances remain NULL. The output is limited to the first 200 role
rows; change the limit when you need a longer history.


In [26]:
%%bq_submit account_history

DECLARE selected_entity_id STRING DEFAULT NULL;
-- Example edit: SET selected_entity_id = 'an-actual-origin-id';

SET selected_entity_id = COALESCE(selected_entity_id, (
    SELECT entity_id
    FROM `amlnet.entity_balance_audit`
    GROUP BY entity_id
    ORDER BY COUNTIF(rolling_status = 'mismatch') DESC, entity_id
    LIMIT 1
));

SELECT
    e.entity_id,
    e.transaction_id,
    e.transaction_timestamp,
    e.transaction_role,
    e.counterparty_id,
    e.counterparty_in_entity_population,
    e.transaction_type,
    e.amount,
    e.signed_amount,
    e.balance_before,
    e.balance_after,
    a.theoretical_balance_before,
    a.theoretical_balance_after,
    a.rolling_difference_before,
    a.interval_difference,
    a.transaction_status,
    a.rolling_status,
    e.label_is_fraud,
    e.label_is_money_laundering
FROM `amlnet.entity_transactions` AS e
LEFT JOIN `amlnet.entity_balance_audit` AS a
    ON e.entity_id = a.entity_id
    AND e.transaction_id = a.transaction_id
    AND e.transaction_role = 'origin'
WHERE e.entity_id = selected_entity_id
ORDER BY e.transaction_timestamp, e.transaction_id, e.transaction_role
LIMIT 200;


Submitting account_history: notebook_6f66cbbba4514e2e99b7d7d78965b339
Submitted. Run the completion-check cell next.


#### Confirm completion: `account_history`

Run this check before continuing. If the job is still running, rerun
**only this cell** shortly. A server-side failure is raised as an error.


In [27]:
account_history_job = check_bq_job("account_history")


account_history: DONE | notebook_6f66cbbba4514e2e99b7d7d78965b339
Completed successfully.


#### Display the selected account's history

Retrieve the completed script's final SELECT through the Python client,
bypassing notebook magic and the Storage API. This reads up to the same
200 rows requested by the SQL; it does not rerun the script. If retrieval
times out, rerun this cell instead of the submission cell.


In [28]:
history_rows = account_history_job.result(
    max_results=200,
    timeout=20,
    retry=None,
    job_retry=None,
)
account_history = pd.DataFrame(
    [dict(row) for row in history_rows],
    columns=[field.name for field in history_rows.schema],
)
display(account_history)


,entity_id,transaction_id,transaction_timestamp,transaction_role,counterparty_id,counterparty_in_entity_population,transaction_type,amount,signed_amount,balance_before,balance_after,theoretical_balance_before,theoretical_balance_after,rolling_difference_before,interval_difference,transaction_status,rolling_status,label_is_fraud,label_is_money_laundering
0,C8888,1710716,2025-10-17 01:47:31.636449+00:00,origin,C1108,True,BPAY,74.964873842,-74.964873842,446848.148726052,446773.183852209,None,None,None,None,within_one_cent,anchor,False,False
1,C8888,1182432,2025-10-17 04:19:47.636449+00:00,origin,C1108,True,OSKO,107.262847402,-107.262847402,446590.237864781,446482.975017379,446773.183852209,446665.921004807,-182.945987428,-182.945987428,exact_match,mismatch,False,False
2,C8888,1803312,2025-10-17 07:44:13.636449+00:00,origin,C2898,True,BPAY,126.634187356,-126.634187356,441772.483909302,441645.849721946,446665.921004807,446539.286817451,-4893.437095505,-4710.491108077,exact_match,mismatch,False,False
3,C8888,1249980,2025-10-17 08:22:44.636449+00:00,origin,M134,False,EFTPOS,2777.967327715,-2777.967327715,450603.537865529,447825.570537813,446539.286817451,443761.319489736,4064.251048078,8957.688143583,within_one_cent,mismatch,False,False
4,C8888,1163680,2025-10-17 12:56:15.636449+00:00,origin,C2898,True,OSKO,239.091390523,-239.091390523,447825.570537813,447586.47914729,443761.319489736,443522.228099213,4064.251048077,0,exact_match,mismatch,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,C8888,1179558,2026-01-17 05:50:51.636449+00:00,origin,C8397,True,EFTPOS,79.447437113,-79.447437113,340615.816291071,340536.368853958,340953.340944011,340873.893506898,-337.52465294,-4322.681335675,exact_match,mismatch,False,False
196,C8888,1773881,2026-01-17 09:05:22.636449+00:00,origin,M898,False,DEBIT,2389.48957806,-2389.48957806,343194.190550594,340804.700972534,340873.893506898,338484.403928838,2320.297043696,2657.821696636,exact_match,mismatch,False,False
197,C8888,1194131,2026-01-18 17:19:34.636449+00:00,origin,M968,False,DEBIT,1544.602787636,-1544.602787636,340536.368853958,338991.766066321,338484.403928838,336939.801141202,2051.96492512,-268.332118576,within_one_cent,mismatch,False,False
198,C8888,1771037,2026-01-21 07:26:37.636449+00:00,origin,C1108,True,NPP,51.649248675,-51.649248675,338913.75886431,338862.109615635,336939.801141202,336888.151892527,1973.957723108,-78.007202011,exact_match,mismatch,False,False


**Observation**

The saved example selects C8888. Its first displayed outflow ends with a balance
of 446,773.18; its next event is another outflow beginning at 446,590.24. With no
intervening recorded event, approximately 182.95 is unexplained by observed flows.
This illustrates the aggregate finding but does not establish its cause. Only
the first 200 role rows are retrieved, and the displayed DataFrame is truncated.


## Assessment and dashboard decisions

The assessment finds that merchant information cannot reliably describe transaction recipients, while recorded account balances are internally consistent within individual transactions but fail to reconcile across time.

| Assessment | Finding | Dashboard decision |
|---|---|---|
| `merchant_id` vs. `destination_id` | No consistent one-to-one mapping exists between the two identifiers. | Exclude `merchant_id` and retain `destination_id` as the recorded recipient. |
| Merchant risk-level consistency | 57 of 1,000 merchant IDs have multiple recorded risk levels, with no verified explanation for the variation. | Exclude `merchant_risk_level` from the dashboard. |
| Within-transaction balance consistency | All transactions satisfy the balance arithmetic check within the 0.01 tolerance. | The arithmetic is valid within individual transactions, but this alone does not establish the reliability of account histories. |
| Temporal balance consistency | 80.99% of consecutive-observation comparisons and 99.52% of first-anchor comparisons fail the 0.01 tolerance. | Exclude account balance histories and balance-derived metrics from the dashboard. |

**Conclusion**

The dataset is suitable for analyzing recorded transaction activity and exploring transactions labelled as fraud or money laundering. However, unreliable merchant information and inconsistent account balances limit its analytical scope. Merchant-related fields and reconstructed account balance histories will therefore be excluded from the dashboard.

The next stage of the project moves to **Power BI**, where an interactive dashboard will be developed with two pages: **Transaction Overview** and **Fraud & AML Connections**, using `amlnet.transactions` as the primary data source.


In [3]:
%%bq_submit cleanup_audit_tables

DROP TABLE IF EXISTS `amlnet.entity_balance_audit`;
DROP TABLE IF EXISTS `amlnet.transaction_balance_audit`;
DROP TABLE IF EXISTS `amlnet.entity_transactions`;
DROP TABLE IF EXISTS `amlnet.account_population`;

Submitting cleanup_audit_tables: notebook_59ffdb932d20427ab7b4731c7e94533d
Submitted. Run the completion-check cell next.


In [4]:
check_bq_job("cleanup_audit_tables")

cleanup_audit_tables: DONE | notebook_59ffdb932d20427ab7b4731c7e94533d
Completed successfully.


QueryJob<project=gen-lang-client-0697353147, location=EU, id=notebook_59ffdb932d20427ab7b4731c7e94533d>